# loading img dataset

In [9]:
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint
import os

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
#pprint(paths_NSD_shared)
#paths_NSD_shared.keys()


stimuli_path = paths_NSD_shared["images"]

## DINOv2

In [4]:
import torch

checkpoint_path = "PATH"

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

print(checkpoint.keys())

dict_keys(['cls_token', 'pos_embed', 'mask_token', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'blocks.0.0.norm1.weight', 'blocks.0.0.norm1.bias', 'blocks.0.0.attn.qkv.weight', 'blocks.0.0.attn.qkv.bias', 'blocks.0.0.attn.proj.weight', 'blocks.0.0.attn.proj.bias', 'blocks.0.0.ls1.gamma', 'blocks.0.0.norm2.weight', 'blocks.0.0.norm2.bias', 'blocks.0.0.mlp.fc1.weight', 'blocks.0.0.mlp.fc1.bias', 'blocks.0.0.mlp.fc2.weight', 'blocks.0.0.mlp.fc2.bias', 'blocks.0.0.ls2.gamma', 'blocks.0.1.norm1.weight', 'blocks.0.1.norm1.bias', 'blocks.0.1.attn.qkv.weight', 'blocks.0.1.attn.qkv.bias', 'blocks.0.1.attn.proj.weight', 'blocks.0.1.attn.proj.bias', 'blocks.0.1.ls1.gamma', 'blocks.0.1.norm2.weight', 'blocks.0.1.norm2.bias', 'blocks.0.1.mlp.fc1.weight', 'blocks.0.1.mlp.fc1.bias', 'blocks.0.1.mlp.fc2.weight', 'blocks.0.1.mlp.fc2.bias', 'blocks.0.1.ls2.gamma', 'blocks.0.2.norm1.weight', 'blocks.0.2.norm1.bias', 'blocks.0.2.attn.qkv.weight', 'blocks.0.2.attn.qkv.bias', 'blocks.0.2.attn.proj.w

In [3]:
import torch
import torch.nn as nn
from functools import partial

checkpoint_path = r"C:\Users\louai\Net2Brain\notebooks\video models\encoders\dinov2_vitb_k710_20pct_iter11399_teacher_backbone.pth"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

print("cls_token:", checkpoint["cls_token"].shape)
print("pos_embed:", checkpoint["pos_embed"].shape)
print("patch_embed:", checkpoint["patch_embed.proj.weight"].shape)
print("block 0 qkv:", checkpoint["blocks.0.0.attn.qkv.weight"].shape)
print("block 11 qkv:", checkpoint["blocks.3.11.attn.qkv.weight"].shape)

cls_token: torch.Size([1, 1, 768])
pos_embed: torch.Size([1, 197, 768])
patch_embed: torch.Size([768, 3, 16, 16])
block 0 qkv: torch.Size([2304, 768])
block 11 qkv: torch.Size([2304, 768])


the architecture exactly: ViT-B/16, 12 blocks, CLS token, 768-dimensional embeddings.

In [4]:
import timm
print(timm.__version__)

test_model = timm.create_model("vit_base_patch16_224", pretrained=False)
print(test_model)

C:\Users\louai\net2brain-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.0.28
VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplac

Great. timm 1.0.28 is already installed, and its vit_base_patch16_224 architecture matches the basic dimensions of your checkpoint: Conv2d 16×16 patching, 768-dimensional embeddings, 12 blocks, qkv 768→2304, and 3072-dimensional MLPs.  
However, there is one mismatch we need to account for: your checkpoint uses DINOv2's LayerScale (ls1.gamma, ls2.gamma), while the plain timm model shown in the output has ls1/ls2 as Identity. So I don't want to load it yet and silently get missing/unmatched weights.

In [5]:
print(test_model.state_dict().keys())

odict_keys(['cls_token', 'pos_embed', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'blocks.0.mlp.fc1.weight', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc2.weight', 'blocks.0.mlp.fc2.bias', 'blocks.1.norm1.weight', 'blocks.1.norm1.bias', 'blocks.1.attn.qkv.weight', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.proj.weight', 'blocks.1.attn.proj.bias', 'blocks.1.norm2.weight', 'blocks.1.norm2.bias', 'blocks.1.mlp.fc1.weight', 'blocks.1.mlp.fc1.bias', 'blocks.1.mlp.fc2.weight', 'blocks.1.mlp.fc2.bias', 'blocks.2.norm1.weight', 'blocks.2.norm1.bias', 'blocks.2.attn.qkv.weight', 'blocks.2.attn.qkv.bias', 'blocks.2.attn.proj.weight', 'blocks.2.attn.proj.bias', 'blocks.2.norm2.weight', 'blocks.2.norm2.bias', 'blocks.2.mlp.fc1.weight', 'blocks.2.mlp.fc1.bias', 'blocks.2.mlp.fc2.weight', 'bloc

In [6]:
print("ls1:", checkpoint["blocks.0.0.ls1.gamma"].shape, checkpoint["blocks.0.0.ls1.gamma"])
print("ls2:", checkpoint["blocks.0.0.ls2.gamma"].shape, checkpoint["blocks.0.0.ls2.gamma"])

ls1: torch.Size([768]) tensor([0.9845, 0.9310, 0.9361, 0.9788, 0.9514, 0.9523, 0.9292, 0.9736, 0.9568,
        0.9120, 0.9521, 0.9199, 0.9496, 0.9336, 0.9399, 0.9192, 0.8723, 0.9117,
        0.9066, 0.9237, 0.9698, 0.9566, 0.9368, 0.9486, 0.9386, 0.9306, 0.9301,
        0.9178, 1.0013, 0.9269, 0.9024, 0.9201, 0.9639, 0.9677, 0.9327, 0.9383,
        0.9363, 0.9337, 0.9097, 0.9644, 0.9481, 0.9785, 0.9205, 0.9344, 0.9381,
        0.9392, 0.9745, 0.9587, 0.9566, 0.9196, 0.9158, 0.9466, 0.9362, 0.9077,
        0.8985, 0.9729, 0.9743, 0.9426, 0.9243, 0.9285, 0.9616, 0.9142, 0.9312,
        0.9317, 0.9739, 0.9097, 0.9169, 0.9093, 0.9438, 0.9081, 0.9665, 0.9456,
        0.9553, 0.9076, 0.9123, 0.9724, 0.9650, 0.8891, 0.9505, 0.9561, 0.9431,
        0.9525, 0.9365, 0.9440, 0.9334, 0.9522, 0.9174, 0.8954, 0.9402, 0.9552,
        0.9466, 0.9267, 0.9379, 0.9795, 0.9085, 0.9202, 0.9042, 0.9283, 0.9661,
        0.9209, 0.9028, 0.9473, 0.9256, 0.8826, 0.9181, 0.9149, 0.9081, 0.9327,
        0.9639, 0

That confirms the LayerScale parameters are learned per feature dimension (768 values each), not a single scalar.  
So we now know the DINOv2 block needs the equivalent of:  
ls1.gamma  
ls2.gamma  
 with shape (768,), and those values must actually be loaded from the checkpoint.

### build the DINOv2 ViT-B/16 with LayerScale and load checkpoint.

In [5]:
import torch
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint_path = "PATH"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0, init_values=1.0) 
#init_vals: because this makes timm create LayerScale modules.

new_state_dict = {}

for key, value in checkpoint.items():
    if key.startswith("blocks."):
        parts = key.split(".")
        if len(parts) >= 4 and parts[1].isdigit() and parts[2].isdigit():
            new_key = "blocks." + parts[2] + "." + ".".join(parts[3:])
            new_state_dict[new_key] = value
        else:
            new_state_dict[key] = value
    elif key != "mask_token": #The mask token is part of the pretraining machinery, not something needed to compute the normal encoder representation for an unmasked image.
        new_state_dict[key] = value

missing, unexpected = model.load_state_dict(new_state_dict, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model = model.to(device)
model.eval()

Missing keys: []
Unexpected keys: []


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=Fal

## Lejepa

In [5]:
lejepa_path = "PATH"

lejepa_checkpoint = torch.load(lejepa_path, map_location="cpu", weights_only=False)

print(lejepa_checkpoint.keys())

dict_keys(['blocks.0.attn.proj.bias', 'blocks.0.attn.proj.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.qkv.weight', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc1.weight', 'blocks.0.mlp.fc2.bias', 'blocks.0.mlp.fc2.weight', 'blocks.0.norm1.bias', 'blocks.0.norm1.weight', 'blocks.0.norm2.bias', 'blocks.0.norm2.weight', 'blocks.1.attn.proj.bias', 'blocks.1.attn.proj.weight', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.qkv.weight', 'blocks.1.mlp.fc1.bias', 'blocks.1.mlp.fc1.weight', 'blocks.1.mlp.fc2.bias', 'blocks.1.mlp.fc2.weight', 'blocks.1.norm1.bias', 'blocks.1.norm1.weight', 'blocks.1.norm2.bias', 'blocks.1.norm2.weight', 'blocks.10.attn.proj.bias', 'blocks.10.attn.proj.weight', 'blocks.10.attn.qkv.bias', 'blocks.10.attn.qkv.weight', 'blocks.10.mlp.fc1.bias', 'blocks.10.mlp.fc1.weight', 'blocks.10.mlp.fc2.bias', 'blocks.10.mlp.fc2.weight', 'blocks.10.norm1.bias', 'blocks.10.norm1.weight', 'blocks.10.norm2.bias', 'blocks.10.norm2.weight', 'blocks.11.attn.proj.bias', 'blocks.11.attn.pro

In [17]:
print("cls_token:", lejepa_checkpoint["cls_token"].shape)
print("pos_embed:", lejepa_checkpoint.get("pos_embed", "NO pos_embed"))
print("patch_embed:", lejepa_checkpoint["patch_embed.proj.weight"].shape)
print("qkv:", lejepa_checkpoint["blocks.0.attn.qkv.weight"].shape)

cls_token: torch.Size([1, 1, 768])
pos_embed: NO pos_embed
patch_embed: torch.Size([768, 3, 1, 16, 16])
qkv: torch.Size([2304, 768])


In [6]:
import torch
import torch.nn as nn
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"

lejepa_model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)

lejepa_model.patch_embed.proj = nn.Conv3d(3, 768, kernel_size=(1, 16, 16), stride=(1, 16, 16), bias=True)

missing, unexpected = lejepa_model.load_state_dict(lejepa_checkpoint, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: ['pos_embed']
Unexpected keys: []


the LeJEPA checkpoint genuinely has no positional-embedding parameter, so the pos_embed missing key is not a missing trained weight. It's a parameter that timm's standard 2D ViT creates but this particular LeJEPA video encoder did not use.

The LeJEPA codebase describes the backbone as a CLS-pooled timm ViT and uses the actual backbone embedding for evaluation, rather than the projector.

So we should construct the backbone with:

CLS token   
12 standard ViT-B blocks  
3D (1,16,16) patch projection  
no positional embedding  

In [9]:
class VideoPatchEmbed1Frame(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv3d(3, 768, kernel_size=(1, 16, 16), stride=(1, 16, 16), bias=True)
        self.norm = nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        x = x.squeeze(2)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x

lejepa_model.patch_embed = VideoPatchEmbed1Frame()
lejepa_model.patch_embed.proj.load_state_dict({
    "weight": lejepa_checkpoint["patch_embed.proj.weight"],
    "bias": lejepa_checkpoint["patch_embed.proj.bias"]
})

print(lejepa_model.patch_embed.proj.weight.shape)

torch.Size([768, 3, 1, 16, 16])


In [10]:
with torch.no_grad():
    x = lejepa_model.patch_embed(x)
    print(x.shape)

torch.Size([1, 196, 768])


In [ ]:
lejepa_model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)
lejepa_model.patch_embed = VideoPatchEmbed1Frame()
lejepa_model.pos_embed = None

missing, unexpected = lejepa_model.load_state_dict(lejepa_checkpoint, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

lejepa_model = lejepa_model.to(device)
lejepa_model.eval()

import torch
import torch.nn as nn
import timm

class VideoPatchEmbed1Frame(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv3d(3, 768, kernel_size=(1, 16, 16), stride=(1, 16, 16), bias=True)
        self.norm = nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        x = x.squeeze(2)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x

lejepa_model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)
lejepa_model.patch_embed = VideoPatchEmbed1Frame()
lejepa_model.pos_embed = None

missing, unexpected = lejepa_model.load_state_dict(lejepa_checkpoint, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

LeJEPA is now fully loaded with zero missing/unexpected keys, so the checkpoint architecture is matched correctly.

### model loading

In [11]:
import os
import torch
import torch.nn as nn
import timm
from net2brain.utils.download_datasets import DatasetAlgonauts_NSD_Shared
from net2brain.feature_extraction import FeatureExtractor
from torchvision import transforms as trn

# =========================
# paths / device
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"
lejepa_path = "PATH"


# =========================
# load checkpoint
# =========================

lejepa_checkpoint = torch.load(lejepa_path, map_location="cpu", weights_only=False)

# =========================
# custom 1-frame video patch embedding
# =========================

class VideoPatchEmbed1Frame(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv3d(3, 768, kernel_size=(1, 16, 16), stride=(1, 16, 16), bias=True) # does what timm does but with [768,3,1,16,16] instead
        self.norm = nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        x = x.squeeze(2)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x

# =========================
# build LeJEPA ViT-B
# =========================

lejepa_model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)
lejepa_model.patch_embed = VideoPatchEmbed1Frame() # using my video patch emb that works with shape [768,3,1,16,16]
lejepa_model.pos_embed = None

missing, unexpected = lejepa_model.load_state_dict(lejepa_checkpoint, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

lejepa_model = lejepa_model.to(device)
lejepa_model.eval()


Missing keys: []
Unexpected keys: []


VisionTransformer(
  (patch_embed): VideoPatchEmbed1Frame(
    (proj): Conv3d(3, 768, kernel_size=(1, 16, 16), stride=(1, 16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0

## video mae

In [12]:
videomae_path = r"C:\Users\louai\Net2Brain\notebooks\video models\encoders\videomae_vitb_k710_20pct_ep239_encoder.pth"

videomae_checkpoint = torch.load(videomae_path, map_location="cpu", weights_only=False)

print("pos_embed:", videomae_checkpoint.get("pos_embed", "NO pos_embed"))
print("patch_embed:", videomae_checkpoint["patch_embed.proj.weight"].shape)
print("q_bias:", videomae_checkpoint["blocks.0.attn.q_bias"].shape)
print("v_bias:", videomae_checkpoint["blocks.0.attn.v_bias"].shape)
print("qkv:", videomae_checkpoint["blocks.0.attn.qkv.weight"].shape)
print("cls_token:", videomae_checkpoint.get("cls_token", "NO cls_token"))

pos_embed: NO pos_embed
patch_embed: torch.Size([768, 3, 2, 16, 16])
q_bias: torch.Size([768])
v_bias: torch.Size([768])
qkv: torch.Size([2304, 768])
cls_token: NO cls_token


No CLS token — consistent with the original VideoMAE encoder; its code explicitly notes that the CLS token is not added.
Tubelet size 2 — an input of 2 frames produces one temporal patch per spatial location.
No learned pos_embed in your checkpoint — the original VideoMAE encoder uses fixed sinusoidal positional encoding rather than a learned parameter.
The encoder has 12 blocks, 768 dimensions, and the usual 4× MLP expansion.

So we shouldn't try to adapt the ordinary 2D timm ViT the same way as LeJEPA. We need to reproduce the VideoMAE encoder's positional encoding and its special q/v-bias attention structure.

In [29]:
for key in videomae_checkpoint:
    if not key.startswith("blocks."):
        print(key, videomae_checkpoint[key].shape if hasattr(videomae_checkpoint[key], "shape") else type(videomae_checkpoint[key]))

patch_embed.proj.weight torch.Size([768, 3, 2, 16, 16])
patch_embed.proj.bias torch.Size([768])
norm.weight torch.Size([768])
norm.bias torch.Size([768])


This confirms your VideoMAE checkpoint is exactly the encoder-only part: patch embedding, 12 transformer blocks, and final norm; there is no CLS token or learned positional embedding.

I also checked the original VideoMAE implementation: the encoder uses fixed sine-cosine positional embeddings, adds them after the tubelet projection, and processes visible patch tokens without a CLS token. Its default base configuration is 16 frames with tubelet size 2.

So for your fMRI experiment, the appropriate image adaptation is:

image → duplicate to 2 identical frames → [B,3,2,224,224] → tubelet projection → fixed positional encoding → 12 blocks → mean-pool tokens.

In [30]:
print("Number of block parameters:", len([k for k in videomae_checkpoint if k.startswith("blocks.")]))
print("Total checkpoint keys:", len(videomae_checkpoint))
print("Patch tubelet size:", videomae_checkpoint["patch_embed.proj.weight"].shape[2])

Number of block parameters: 156
Total checkpoint keys: 160
Patch tubelet size: 2


 loading

In [13]:
import torch
import torch.nn as nn
import math

def get_sinusoid_encoding_table(n_position, d_hid):
    def get_position_angle_vec(position):
        return [position / math.pow(10000, 2 * (hid_j // 2) / d_hid) for hid_j in range(d_hid)]
    sinusoid_table = torch.tensor([get_position_angle_vec(pos_i) for pos_i in range(n_position)], dtype=torch.float32)
    sinusoid_table[:, 0::2] = torch.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = torch.cos(sinusoid_table[:, 1::2])
    return sinusoid_table.unsqueeze(0)

class VideoMAEAttention(nn.Module):
    def __init__(self, dim=768, num_heads=12):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.q_bias = nn.Parameter(torch.zeros(dim))
        self.v_bias = nn.Parameter(torch.zeros(dim))
        self.attn_drop = nn.Dropout(0.0)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(0.0)

    def forward(self, x):
        B, N, C = x.shape
        qkv_bias = torch.cat((self.q_bias, torch.zeros_like(self.v_bias), self.v_bias))
        qkv = torch.nn.functional.linear(x, self.qkv.weight, qkv_bias)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class VideoMAEMLP(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * 4)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(dim * 4, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class VideoMAEBlock(nn.Module):
    def __init__(self, dim=768, num_heads=12):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = VideoMAEAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = VideoMAEMLP(dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class VideoMAEEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = nn.Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16), bias=True)
        self.pos_embed = get_sinusoid_encoding_table(1568, 768)
        self.blocks = nn.ModuleList([VideoMAEBlock(768, 12) for _ in range(12)])
        self.norm = nn.LayerNorm(768, eps=1e-6)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed.to(dtype=x.dtype, device=x.device)
        for block in self.blocks:
            x = block(x)
        return self.norm(x)

videomae_model = VideoMAEEncoder()

new_state_dict = {}
for key, value in videomae_checkpoint.items():
    if key.startswith("patch_embed.proj."):
        new_key = key.replace("patch_embed.proj.", "patch_embed.")
        new_state_dict[new_key] = value
    else:
        new_state_dict[key] = value

missing, unexpected = videomae_model.load_state_dict(new_state_dict, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: []
Unexpected keys: []


In [34]:
# test 1 img
from PIL import Image
from torchvision import transforms

image_path = os.path.join(stimuli_path, os.listdir(stimuli_path)[0])
image = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

frame = transform(image)
x = frame.unsqueeze(0).unsqueeze(2).repeat(1, 1, 2, 1, 1).to(device)

with torch.no_grad():
    features = videomae_model(x)

print("Input:", x.shape)
print("Output:", features.shape)

RuntimeError: The size of tensor a (196) must match the size of tensor b (1568) at non-singleton dimension 1

In [35]:
videomae_model.pos_embed = get_sinusoid_encoding_table(196, 768)
print(videomae_model.pos_embed.shape)

torch.Size([1, 196, 768])


In [36]:
with torch.no_grad():
    features = videomae_model(x)

print("Input:", x.shape)
print("Output:", features.shape)

Input: torch.Size([1, 3, 2, 224, 224])
Output: torch.Size([1, 196, 768])


### final model loading

In [29]:
import os
import math
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# checkpoint
# =========================

videomae_path = "PATH"
videomae_checkpoint = torch.load(videomae_path, map_location="cpu", weights_only=False)

# =========================
# fixed sinusoidal position encoding
# =========================

def get_sinusoid_encoding_table(n_position, d_hid):
    def get_position_angle_vec(position):
        return [position / math.pow(10000, 2 * (hid_j // 2) / d_hid) for hid_j in range(d_hid)]
    sinusoid_table = torch.tensor([get_position_angle_vec(pos_i) for pos_i in range(n_position)], dtype=torch.float32)
    sinusoid_table[:, 0::2] = torch.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = torch.cos(sinusoid_table[:, 1::2])
    return sinusoid_table.unsqueeze(0)

# =========================
# VideoMAE attention
# =========================

class VideoMAEAttention(nn.Module):
    def __init__(self, dim=768, num_heads=12):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.q_bias = nn.Parameter(torch.zeros(dim))
        self.v_bias = nn.Parameter(torch.zeros(dim))
        self.attn_drop = nn.Dropout(0.0)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(0.0)

    def forward(self, x):
        B, N, C = x.shape
        qkv_bias = torch.cat((self.q_bias, torch.zeros_like(self.v_bias), self.v_bias))
        qkv = torch.nn.functional.linear(x, self.qkv.weight, qkv_bias)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

# =========================
# VideoMAE MLP
# =========================

class VideoMAEMLP(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * 4)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(dim * 4, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

# =========================
# VideoMAE transformer block
# =========================

class VideoMAEBlock(nn.Module):
    def __init__(self, dim=768, num_heads=12):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = VideoMAEAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = VideoMAEMLP(dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

# =========================
# VideoMAE encoder
# =========================

class VideoMAEEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = nn.Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16), bias=True)
        self.pos_embed = get_sinusoid_encoding_table(196, 768)
        self.blocks = nn.ModuleList([VideoMAEBlock(768, 12) for _ in range(12)])
        self.norm = nn.LayerNorm(768, eps=1e-6)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed.to(dtype=x.dtype, device=x.device)
        for block in self.blocks:
            x = block(x)
        return self.norm(x)

# =========================
# create model and load weights
# =========================

videomae_model = VideoMAEEncoder()

new_state_dict = {}
for key, value in videomae_checkpoint.items():
    if key.startswith("patch_embed.proj."):
        new_key = key.replace("patch_embed.proj.", "patch_embed.")
        new_state_dict[new_key] = value
    else:
        new_state_dict[key] = value

missing, unexpected = videomae_model.load_state_dict(new_state_dict, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

videomae_model = videomae_model.to(device)
videomae_model.eval()

# =========================
# test one NSD image
# =========================

image_path = os.path.join(stimuli_path, os.listdir(stimuli_path)[0])
image = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

frame = transform(image)
x = frame.unsqueeze(0).unsqueeze(2).repeat(1, 1, 2, 1, 1).to(device)

with torch.no_grad():
    features = videomae_model(x)

print("Input:", x.shape)
print("Output:", features.shape)

Missing keys: []
Unexpected keys: []
Input: torch.Size([1, 3, 2, 224, 224])
Output: torch.Size([1, 196, 768])


## V-JEPA

In [12]:
vjepa_path = "PATH"

vjepa_checkpoint = torch.load(vjepa_path, map_location="cpu", weights_only=False)

print("Non-block keys:")
for key in vjepa_checkpoint:
    if not key.startswith("blocks."):
        value = vjepa_checkpoint[key]
        print(key, value.shape if hasattr(value, "shape") else type(value))

print("Block 0 qkv:", vjepa_checkpoint["blocks.0.attn.qkv.weight"].shape)

Non-block keys:
pos_embed torch.Size([1, 1568, 768])
patch_embed.proj.weight torch.Size([768, 3, 2, 16, 16])
patch_embed.proj.bias torch.Size([768])
norm.weight torch.Size([768])
norm.bias torch.Size([768])
Block 0 qkv: torch.Size([2304, 768])


pos_embed      [1, 1568, 768]  
patch_embed    [768, 3, 2, 16, 16]  
norm           [768]  
it is a video ViT-B with 16-frame training input, tubelet size 2, 14×14 spatial patches, and no CLS token:  
16 / 2 = 8 temporal positions  
8 × 14 × 14 = 1568 tokens  
Unlike VideoMAE, it has a learned positional embedding

In [13]:
print("qkv bias:", vjepa_checkpoint.get("blocks.0.attn.qkv.bias", "NO qkv bias"))
print("q bias:", vjepa_checkpoint.get("blocks.0.attn.q_bias", "NO q_bias"))
print("v bias:", vjepa_checkpoint.get("blocks.0.attn.v_bias", "NO v_bias"))
print("ls1:", vjepa_checkpoint.get("blocks.0.ls1.gamma", "NO ls1"))
print("ls2:", vjepa_checkpoint.get("blocks.0.ls2.gamma", "NO ls2"))

qkv bias: tensor([-0.0081,  0.0062, -0.0093,  ..., -0.0451,  0.0356, -0.0177])
q bias: NO q_bias
v bias: NO v_bias
ls1: NO ls1
ls2: NO ls2


This is the simplest transformer block of the three:  

standard qkv with a single learned bias  
no q_bias / v_bias split  
no LayerScale   
12 ViT-B blocks  
3D patch embedding with temporal kernel 2  
learned pos_embed  
no CLS token  

In [14]:
print("norm1:", vjepa_checkpoint["blocks.0.norm1.weight"].shape)
print("attn proj:", vjepa_checkpoint["blocks.0.attn.proj.weight"].shape)
print("mlp fc1:", vjepa_checkpoint["blocks.0.mlp.fc1.weight"].shape)
print("mlp fc2:", vjepa_checkpoint["blocks.0.mlp.fc2.weight"].shape)

norm1: torch.Size([768])
attn proj: torch.Size([768, 768])
mlp fc1: torch.Size([3072, 768])
mlp fc2: torch.Size([768, 3072])


Perfect. That confirms the transformer blocks are the ordinary ViT-B dimensions:  

768 → attention → 768  
768 → 3072 → 768  
12 blocks  

In [15]:
import torch
import torch.nn as nn
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"

class VideoPatchEmbed2Frame(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16), bias=True)
        self.norm = nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x

vjepa_model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)

vjepa_model.patch_embed = VideoPatchEmbed2Frame()
vjepa_model.pos_embed = nn.Parameter(torch.zeros(1, 1568, 768))

missing, unexpected = vjepa_model.load_state_dict(vjepa_checkpoint, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: ['cls_token']
Unexpected keys: []


That is expected. timm creates a cls_token by default, but your V-JEPA checkpoint has no CLS token.

So this isn't a checkpoint problem. We just need to remove the unused cls_token parameter from the model.

In [16]:
vjepa_model.cls_token = None

missing, unexpected = vjepa_model.load_state_dict(vjepa_checkpoint, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: []
Unexpected keys: []


In [17]:
from PIL import Image
from torchvision import transforms

image_path = os.path.join(stimuli_path, os.listdir(stimuli_path)[0])
image = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

frame = transform(image)
x = frame.unsqueeze(0).unsqueeze(2).repeat(1, 1, 2, 1, 1).to(device)

print("Input:", x.shape)

Input: torch.Size([1, 3, 2, 224, 224])


In [19]:
import math
import torch

def get_sinusoid_encoding_table(n_position, d_hid):
    def get_position_angle_vec(position):
        return [position / math.pow(10000, 2 * (hid_j // 2) / d_hid) for hid_j in range(d_hid)]
    sinusoid_table = torch.tensor([get_position_angle_vec(pos_i) for pos_i in range(n_position)], dtype=torch.float32)
    sinusoid_table[:, 0::2] = torch.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = torch.cos(sinusoid_table[:, 1::2])
    return sinusoid_table.unsqueeze(0)
    
pos_1568 = vjepa_checkpoint["pos_embed"]
pos_196_sincos = get_sinusoid_encoding_table(196, 768)

print("checkpoint first 196:", pos_1568[:, :196, :].shape)
print("generated:", pos_196_sincos.shape)
print("max difference:", (pos_1568[:, :196, :] - pos_196_sincos).abs().max().item())

checkpoint first 196: torch.Size([1, 196, 768])
generated: torch.Size([1, 196, 768])
max difference: 2.0


before testing the model. My earlier comparison was wrong because the helper we used for VideoMAE was a 1D positional encoding over flattened positions, while V-JEPA uses a 3D spatiotemporal sinusoidal positional encoding.

The official V-JEPA implementation constructs the encoder positional embedding with get_3d_sincos_pos_embed, using separate temporal, height, and width coordinates. It also handles a shorter video by taking the first required positions when the spatial resolution is unchanged.

So for your V-JEPA:

original:
8 temporal × 14 height × 14 width = 1568

our input:
1 temporal × 14 height × 14 width = 196

We should not generate a generic 196-position 1D encoding.

In [20]:
from src.models.utils.pos_embs import get_3d_sincos_pos_embed

print(get_3d_sincos_pos_embed)

ModuleNotFoundError: No module named 'src'

the official code gives us the answer, and it corrects an important assumption from earlier.

Your V-JEPA pos_embed is a fixed 3D sinusoidal positional encoding stored in the checkpoint, not a learned embedding. The official implementation creates it with get_3d_sincos_pos_embed(...), sets requires_grad=False, and for a shorter temporal input it takes the first new_N positions when height/width are unchanged.

So for your exact experiment, we should not regenerate a new 196-position encoding and we should not compare it to our VideoMAE 1D encoding.

For your 2-frame input:

2 frames / tubelet 2 = 1 temporal position
14 × 14 spatial positions
→ 196 tokens

In [21]:
pos_embed = vjepa_checkpoint["pos_embed"]

print("Full:", pos_embed.shape)
print("2-frame slice:", pos_embed[:, :196, :].shape)
print("requires_grad:", getattr(pos_embed, "requires_grad", "checkpoint tensor"))
print("first position:", pos_embed[0, 0, :5])
print("196th position:", pos_embed[0, 195, :5])

Full: torch.Size([1, 1568, 768])
2-frame slice: torch.Size([1, 196, 768])
requires_grad: False
first position: tensor([0., 0., 0., 0., 0.])
196th position: tensor([0., 0., 0., 0., 0.])


### model loading

In [3]:
import torch
import torch.nn as nn
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# checkpoint
# =========================

vjepa_path = r"C:\Users\louai\Net2Brain\notebooks\video models\encoders\vjepa_vitb_k710_20pct_ep240_target_encoder.pth"
vjepa_checkpoint = torch.load(vjepa_path, map_location="cpu", weights_only=False)

# =========================
# 2-frame video patch embedding
# =========================

class VideoPatchEmbed2Frame(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16), bias=True)
        self.norm = nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x

# =========================
# build ViT-B
# =========================

vjepa_model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)

vjepa_model.patch_embed = VideoPatchEmbed2Frame()
vjepa_model.pos_embed = nn.Parameter(torch.zeros(1, 1568, 768), requires_grad=False)
vjepa_model.cls_token = None

# =========================
# load checkpoint
# =========================

missing, unexpected = vjepa_model.load_state_dict(vjepa_checkpoint, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

# =========================
# use only the required
# positional embeddings for
# the 2-frame input
# =========================

vjepa_model.pos_embed = nn.Parameter(
    vjepa_checkpoint["pos_embed"][:, :196, :].clone(),
    requires_grad=False
)

vjepa_model = vjepa_model.to(device)
vjepa_model.eval()

print("Positional embedding:", vjepa_model.pos_embed.shape)

C:\Users\louai\net2brain-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Missing keys: []
Unexpected keys: []
Positional embedding: torch.Size([1, 196, 768])


# feature extracion

In [7]:
from torchvision import transforms as trn
import torchextractor as tx
from net2brain.feature_extraction import FeatureExtractor
import torch


# =========================
# preprocessing
# =========================

def my_preprocessor(image, model_name, device):
    """
    Args:
        image (Union[Image.Image, List[Image.Image]]): A PIL Image or a list of PIL Images.
        model_name (str): The name of the model, used to determine specific preprocessing if necessary.
        device (str): The device to which the tensor should be transferred ('cuda' for GPU, 'cpu' for CPU).

    Returns:
        Union[torch.Tensor, List[torch.Tensor]]: The preprocessed image(s) as PyTorch tensor(s).
    """

    transforms = trn.Compose([
        trn.Resize((224, 224)),
        trn.ToTensor(),
        trn.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])

    img_tensor = transforms(image).unsqueeze(0)

    if device == 'cuda':
        img_tensor = img_tensor.cuda()

    return img_tensor

# =========================
# cleaner
# =========================

def my_cleaner(features):
    return features

### Dino

In [2]:
import torch.nn as nn

def my_video_dino_extractor(preprocessed_data, layers_to_extract, model):
    dino = model.model
    features = {}

    with torch.no_grad():
        x = dino.patch_embed(preprocessed_data)
        x = dino._pos_embed(x)
        x = dino.patch_drop(x)
        x = dino.norm_pre(x)

        for i, block in enumerate(dino.blocks):
            x = block(x)
            # mean pooling
            #pooled = x[:, 1:, :].mean(dim=1)
            #features[f"layer_{i+1}"] = pooled
            # CLS
            features[f"layer_{i+1}"] = x[:, 0, :]
            
        
        x = dino.norm(x)
        # mean pooling
        #pooled = x[:, 1:, :].mean(dim=1)
        #features["layer_13"] = pooled
        # CLS:
        features["layer_13"] = x[:, 0, :]

    return features

In [7]:
import torch.nn as nn
class DINOFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

dino_wrapper = DINOFeatureExtractor(model)

In [8]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(
    model=dino_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_video_dino_extractor
)

In [11]:
fx.extract(
    data_path=stimuli_path,
    save_path="PATH"
    consolidate_per_layer=True
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [07:04<00:00,  2.06it/s, subfiles=1/1]


Consolidating data per layer...


100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [01:20<00:00, 10.90it/s]


### Lejepa

In [20]:
import torch.nn as nn


def my_lejepa_extractor(preprocessed_data, layers_to_extract, model):
    lejepa = model.model
    features = {}

    with torch.no_grad():
        x = preprocessed_data.unsqueeze(2)
        x = lejepa.patch_embed(x)
        x = torch.cat((lejepa.cls_token.expand(x.shape[0], -1, -1), x), dim=1)

        for i, block in enumerate(lejepa.blocks):
            x = block(x)
            # mean pooling
            #pooled = x[:, 1:, :].mean(dim=1)
            #features[f"layer_{i+1}"] = pooled
            # CLS
            features[f"layer_{i+1}"] = x[:, 0, :]

        x = lejepa.norm(x)
        # mean pooling
        #pooled = x[:, 1:, :].mean(dim=1)
        #features["layer_13"] = pooled

        #CLS
        features[f"layer_{i+1}"] = x[:, 0, :]

    return features

In [21]:
import torch.nn as nn

class LeJEPAFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

le_jepa_wrapper = LeJEPAFeatureExtractor(lejepa_model)

In [22]:

from net2brain.feature_extraction import FeatureExtractor
fx_lejepa = FeatureExtractor(
    model=le_jepa_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_lejepa_extractor
)

fx_lejepa.extract(
    data_path=stimuli_path,
    save_path=r"C:\Users\louai\Net2Brain\notebooks\0_FEATURE EXTRACTIONS\LeVJEPA_Feat_nsd_unconsolidated",
    consolidate_per_layer=False
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [06:58<00:00,  2.08it/s, subfiles=1/1]


### maevideo

In [30]:
import torch.nn as nn

class VideoMAEFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

videomae_wrapper = VideoMAEFeatureExtractor(videomae_model)


def my_videomae_extractor(preprocessed_data, layers_to_extract, model):
    videomae = model.model
    features = {}

    with torch.no_grad(): #we arent using the model's normal forward pass
        x = preprocessed_data.unsqueeze(2).repeat(1, 1, 2, 1, 1)
        x = videomae.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        x = x + videomae.pos_embed.to(dtype=x.dtype, device=x.device)

        for i, block in enumerate(videomae.blocks):
            x = block(x) #Take the current representation and pass it through one complete transformer block.
            features[f"layer_{i+1}"] = x.mean(dim=1)

        x = videomae.norm(x)
        features["layer_13"] = x.mean(dim=1)

    return features

In [33]:

from net2brain.feature_extraction import FeatureExtractor
fx_videomae = FeatureExtractor(
    model=videomae_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_videomae_extractor
)

fx_videomae.extract(
    data_path=stimuli_path,
    save_path=r'C:\Users\louai\Net2Brain\notebooks\0_FEATURE EXTRACTIONS\VIDEOMAE_Feat_nsd',
    consolidate_per_layer=True
)

100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [01:10<00:00, 12.40it/s]


### VJEPA

In [4]:
import torch.nn as nn

class VJEPAFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

vjepa_wrapper = VJEPAFeatureExtractor(vjepa_model)


def my_vjepa_extractor(preprocessed_data, layers_to_extract, model):
    vjepa = model.model
    features = {}

    with torch.no_grad():
        x = preprocessed_data.unsqueeze(2).repeat(1, 1, 2, 1, 1)
        x = vjepa.patch_embed(x)
        x = x + vjepa.pos_embed[:, :x.shape[1], :].to(dtype=x.dtype, device=x.device)

        for i, block in enumerate(vjepa.blocks):
            x = block(x)
            features[f"layer_{i+1}"] = x.mean(dim=1)

        x = vjepa.norm(x)
        features["layer_13"] = x.mean(dim=1)

    return features

In [11]:
from net2brain.feature_extraction import FeatureExtractor
fx_vjepa = FeatureExtractor(
    model=vjepa_wrapper,
    device=device,
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_vjepa_extractor
)

fx_vjepa.extract(
    data_path=stimuli_path,
    save_path=r"C:\Users\louai\Net2Brain\notebooks\0_FEATURE EXTRACTIONS\VJEPA_Feat_nsd",
    consolidate_per_layer=True
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [07:02<00:00,  2.06it/s, subfiles=1/1]


Consolidating data per layer...


100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [01:17<00:00, 11.30it/s]
